<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/06_generate_sentiment_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generating Financial Sentiment Explanations

In this notebook, we move beyond simple sentiment labels to create a **reasoned dataset**. We take our augmented dataset (original and synthetic headlines) and generate high-quality financial explanations for each.

### Why generate explanations?
- **Chain-of-Thought (CoT) Fine-tuning**: Training a model to reason before providing a label significantly improves its accuracy and robustness.
- **Transparency**: In financial applications, an "explainable" AI is more trustworthy for human analysts.
- **Data Augmentation**: We use a powerful teacher model (**Qwen 2.5 7B**) to "teach" a smaller student model how to think like a senior financial analyst.

### Technical Approach
We utilize **4-bit NormalFloat (NF4) quantization** to run a 7-billion parameter model on a single consumer GPU, ensuring high performance with manageable hardware requirements.

In [ ]:
%%capture
import os
if "COLAB_" in "".join(os.environ.keys()):
    !pip install -U transformers trl accelerate bitsandbytes

In [ ]:
import torch
from datasets import load_from_disk, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

## 1. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID = "FinancialPhraseBank_augmented_judged"
OUTPUT_PATH = "FinancialPhraseBank_explained"
BATCH_SIZE = 16
MAX_NEW_TOKENS = 512

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

## 2. Prompt Engineering

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior financial analyst with deep expertise in equity markets, "
    "corporate finance, and macroeconomics. "
    "Your task is to explain, in 2-4 sentences, why a financial news headline "
    "carries a specific market sentiment. "
    "Focus strictly on the financial implications: how the news affects revenue, "
    "profitability, cash flow, investor confidence, or market positioning. "
    "Be concise and precise. Do not repeat the sentence verbatim."
)

def build_prompt(sentence: str, sentiment: str) -> str:
    return (
        f'Financial news headline:\n"{sentence}"\n\n'
        f"This headline has been classified as **{sentiment}** sentiment "
        f"from a financial markets perspective.\n\n"
        f"Explain why, focusing on the financial implications for investors, "
        f"the company, or the broader market."
    )

## 3. Loading the Teacher Model

In [ ]:
def load_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model

tokenizer, model = load_model(MODEL_ID)

## 4. Inference Pipeline

In [ ]:
def generate_explanations_batch(tokenizer, model, sentences, labels):
    chats = [
        tokenizer.apply_chat_template([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(s, LABEL_MAP[l])},
        ], tokenize=False, add_generation_prompt=True)
        for s, l in zip(sentences, labels)
    ]
    
    inputs = tokenizer(chats, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs["input_ids"].shape[1]
    explanations = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    return [e.strip() for e in explanations]

## 5. Dataset Processing

In [ ]:
def process_dataset(tokenizer, model, dataset):
    results = []
    sentences = dataset["sentence"]
    labels = dataset["label"]
    is_augmented = dataset["is_augmented"]
    judge_score = dataset["judge_score"]
    
    for start in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Generating explanations"):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels = labels[start : start + BATCH_SIZE]
        
        explanations = generate_explanations_batch(tokenizer, model, batch_sentences, batch_labels)
        
        for i, (sentence, label, explanation) in enumerate(zip(batch_sentences, batch_labels, explanations)):
            idx = start + i
            results.append({
                "sentence": sentence,
                "label": label,
                "sentiment": LABEL_MAP[label],
                "explanation": explanation,
                "is_augmented": is_augmented[idx],
                "judge_score": judge_score[idx]
            })
    return results

## 6. Execution

In [ ]:
print(f"Loading augmented dataset from: {DATASET_ID}")
raw = load_from_disk(DATASET_ID)

output_splits = {}
for split_name, data in raw.items():
    print(f"\n── Processing split: {split_name} ──")
    results = process_dataset(tokenizer, model, data)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nDataset saved to: {OUTPUT_PATH}")

## 7. Verification

In [ ]:
ex = output_ds["train"][0]
print(f"Sentence  : {ex['sentence']}")
print(f"Augmented : {ex['is_augmented']}")
print(f"Explanation: {ex['explanation']}")